In [5]:
import glob
import pandas as pd
import re
from pathlib import Path
from tqdm import tqdm
import pickle 
import sys, os
from openai import AzureOpenAI
from utils.data_processing import load_lexicon, add_clean_descriptors_column, remove_parentheses_text, DEFAULT_ADJECTIVES
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv() 

True

In [6]:
# Set up azure API client
subscription_key = os.getenv("AZURE_API_KEY")
endpoint = os.getenv("AZURE_ENDPOINT")
model_name = os.environ["MODEL_NAME"]
deployment = os.environ["DEPLOYMENT"]
api_version = os.environ["API_VERSION"]

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [7]:
# Test for the API

common_name = "sumac"
scientific_name = "Rhus typhina"
retrieved_nodes = ""

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": """
            You are a food science expert.
            Your task is to extract sensory flavor adjectives.
            Return a comma-separated list.
            """
            },
            {
            "role": "user",
            "content": f"""
            Ingredient:
            Common name: {common_name}
            Scientific name: {scientific_name}
            """
            }
    ],
    max_completion_tokens=16384,
    model=deployment
)

print(response.choices[0].message.content)

tangy, tart, sour, lemony, citrusy, bright, slightly fruity


# Load data

In [4]:
files = glob.glob("data/RAG/*.out")
path = files[0]

In [5]:
# Native RAG extraction 
text = Path(path).read_text(errors="ignore")

# 1) Drop any preamble noise before the FIRST [PROMPT]
end = text.find("Running Flavor_DB ingredients\n")
text = text[:end]

chunks = text.split("[RAG]")

# Prompt regex (full prompt)
prompt_re = re.compile(
    r"(What does\s+.+?,\s+with the scientific name\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# Scientific name regex (captures ONLY the scientific name)
sci_name_re = re.compile(
    r"with the scientific name\s+([^,]+)",
    flags=re.I
)

# Common name regex
common_name_re = re.compile(
    r"What does\s+([^,]+)",
    flags=re.I
)

# Regex to capture everything between [RAG] and the next prompt
rag_block_re = re.compile(
    r"\[RAG\](.*?)(?=" + prompt_re.pattern + r")",
    flags=re.I | re.S
)

records = []
current_prompt = None
current_sci_name = None


for rag_match in rag_block_re.finditer(text):
    rag_block = rag_match.group(1).strip()

    # Extract prompt from this region
    m_prompt = prompt_re.search(text, rag_match.end())
    if not m_prompt:
        continue

    prompt = m_prompt.group(1).strip()

    # Extract scientific name
    m_sci = sci_name_re.search(prompt)
    sci_name = m_sci.group(1).strip() if m_sci else None

    m_common = common_name_re.search(prompt)
    current_common = m_common.group(1).strip() if m_common else None

    # Extract final answer after this prompt
    m_answer = re.search(
        r"\[FINAL ANSWER\](.*?)(?=\[RAG\]|\Z)",
        text[m_prompt.end():],
        flags=re.I | re.S
    )

    answer = m_answer.group(1).strip() if m_answer else None

    records.append({
        "prompt": prompt,
        "common_name": current_common,
        "scientific_name": sci_name,
        "rag_text": rag_block,   # <- text between [RAG] and prompt
    })

In [7]:
# Text cleaning
pattern = r"--- Chunk \d+ \(Score: [\d.]+\) ---\r?\nSource: .*?\.txt\r?\n?"

for record in records:
    rag_text = record.get("rag_text", "")
    
    cleaned_text = re.sub(pattern, "", rag_text)
    cleaned_text = cleaned_text.replace("...\n", "\n")
    
    record["rag_text"] = cleaned_text


In [9]:
# API call for native inredient extraction

responses = []
for record in tqdm(records):
    common_name = record['common_name']
    scientific_name = 'scientific_name'
    retrieved_nodes = 'rag_text'

    system_prompt = (
            "You are a food science expert. "
            "Your task is to extract sensory flavor adjectives. "
            "Return a comma-separated list."
        )
    user_prompt = (
        f"Ingredient:\n"
        f"Common name: {common_name}\n"
        f"Scientific name: {scientific_name}\n"
        "Reference context:\n"
        f"{retrieved_nodes}\n"
        "Return a comma-separated list of flavor adjectives best describing the above ingredient."
    )

    response = client.chat.completions.create(
        messages=[
        {
            "role": "system",
            "content": system_prompt
            },
            {
            "role": "user",
            "content": user_prompt
            }
        ],
        max_completion_tokens=16384,
        model=deployment,
    )
    responses.append(response)
    record["answer"] = response.choices[0].message.content
    record['sys_prompt'] = system_prompt
    record['user_prompt'] = user_prompt

    with open("./data/gpt-5_responses.pkl", "wb") as f:
        pickle.dump(responses, f)

    # print(response.choices[0].message.content)

  0%|          | 0/168 [00:00<?, ?it/s]

  0%|          | 0/168 [00:03<?, ?it/s]


In [155]:
df = pd.DataFrame.from_records(records)
df.to_csv("data/gpt-52_native_extraction.csv", index=False)

## merging ai azure values with data from pipeline

In [ ]:
df = pd.read_csv('data/local_df_ai_descriptors.csv')
df1 = pd.read_csv("data/gpt-52_native_extraction.csv")

df = df.dropna(subset=['Nom scientifique'])

df['scientific_name'] = df['Nom scientifique'].str.lower().str.strip()
df1['scientific_name'] = df1['scientific_name'].str.lower().str.strip()


In [ ]:
# # QC
# df['Nom scientifique'].unique().shape
# df1['scientific_name'].unique().shape
# # looking for specific vals
# s = "polyporus spp."
# df.loc[df['Nom scientifique'].str.contains(s, case=False, na=False)]
# df1.loc[df1['scientific_name'].str.contains(s, case=False, na=False)]

# # looking for vals in df1 that are not in df
# df_diff_1 = df.merge(
#     df1,
#     left_on="scientific_name",
#     right_on="scientific_name",
#     how="outer",
#     indicator=True
# )
# df_diff_1[df_diff_1["prompt"].isna()]

,Unnamed: 0,group,Nom scientifique,Nom vernaculaire,Famille botanique,Partie comestible,Méthodes de consommation,Caractéristiques sensorielles,Valeurs nutritionnelles,Composes toxique,...,categorie,category,scientific_name,prompt,common_name,rag_text,answer,sys_prompt,user_prompt,_merge


In [ ]:
df_merged = df.merge(
    df1,
    left_on="scientific_name",
    right_on="scientific_name",
    how="left"
)
df_merged.drop(columns=['Unnamed: 0'], inplace=True)


In [ ]:
df_merged.to_csv('data/local_df_ai_descriptors.csv', index=False)

# Flavordb extraction

In [156]:
df1 = pd.read_csv('data/merged_ai_descriptors_clean.csv')
df1 = df1.loc[df1['db']=='flavor_db']
flavordb_ing = df1['Nom scientifique'].to_list()

In [161]:
# API call for flavorDB ingredients

responses = []
records = []
for ing in tqdm(flavordb_ing):
    record ={}
    record['common_name'] = ing

    system_prompt = (
            "You are a food science expert. "
            "Your task is to extract sensory flavor adjectives. "
            "Return a comma-separated list."
        )
    user_prompt = (
            f"Ingredient:\n"
            f"Common name: {ing}\n"
            "Return a comma-separated list of flavor adjectives best describing the above ingredient."
        )

    response = client.chat.completions.create(
        messages=[
        {
            "role": "system",
            "content": system_prompt
            },
            {
            "role": "user",
            "content": user_prompt
            }
        ],
        max_completion_tokens=16384,
        model=deployment,
    )
    responses.append(response)

    record["answer"] = response.choices[0].message.content
    record['sys_prompt'] = system_prompt
    record['user_prompt'] = user_prompt
    records.append(record)

    with open("./data/gpt-5_flavordb.pkl", "wb") as f:
        pickle.dump(responses, f)

    # print(response.choices[0].message.content)

100%|██████████| 934/934 [1:18:09<00:00,  5.02s/it]


In [162]:
df1 = pd.DataFrame.from_records(records)
df1.to_csv("data/gpt-52_flavordb_extraction.csv", index=False)

## merging ai azure values with data from pipeline

In [376]:
flavor_db = pd.read_csv('data/flavor_db_descriptors.csv')
df1 =  pd.read_csv('data/gpt-52_flavordb_extraction.csv')

print(flavor_db.shape)
print(df1.shape)

In [ ]:
# QC
merged_df = flavor_db.merge(
    df1,
    right_on="common_name",
    left_on="entity_alias_readable",
    how="left",
    indicator=True
)

missing_in_df1 = merged_df[merged_df["_merge"] == "left_only"]

missing_in_df1

reverse_merge = flavor_db.merge(
    df1,
    right_on="common_name",
    left_on="entity_alias_readable",
    how="right",
    indicator=True
)

missing_in_flavor_db = reverse_merge[reverse_merge["_merge"] == "right_only"]

missing_in_flavor_db

,category,entity_id,category_readable,entity_alias_basket,entity_alias_readable,natural_source_name,entity_alias,molecules,natural_source_url,entity_alias_url,entity_alias_synonyms,error,sub_category,descriptors,common_name,answer,sys_prompt,user_prompt,_merge


In [ ]:
merged_df = flavor_db.merge(df1, right_on='common_name', left_on='entity_alias_readable', how='left')
merged_df.head()

In [375]:
merged_df.to_csv('data/flavor_db_descriptors.csv', index=False)

# Misc

In [10]:
# all ingredients (both native and flavorDB)

path = files[0]
text = Path(path).read_text(errors="ignore")
# 1) Drop any preamble noise before the FIRST [PROMPT]
end = text.find("Running Flavor_DB ingredients\n")
text = text[end:]


chunks = text.split("[PROMPT]")

# ---- Prompt patterns ----

# 1) With scientific name
prompt_sci_re = re.compile(
    r"(What does\s+.+?,\s+with the scientific name\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# 2) Common name only
prompt_common_re = re.compile(
    r"(What does\s+.+?,\s+taste like\?\s+Give a comma-separated list of flavor adjectives\.)",
    flags=re.I | re.S
)

# Extractors
sci_name_re = re.compile(
    r"with the scientific name\s+([^,]+)",
    flags=re.I
)

common_name_re = re.compile(
    r"What does\s+([^,]+)",
    flags=re.I
)

records = []
current_prompt = None
current_common = None
current_sci = None

for chunk in chunks:
    # --- Try scientific-name prompt first ---
    m = prompt_sci_re.search(chunk)
    if m:
        current_prompt = m.group(1).strip()

        m_sci = sci_name_re.search(current_prompt)
        current_sci = m_sci.group(1).strip() if m_sci else None

        m_common = common_name_re.search(current_prompt)
        current_common = m_common.group(1).strip() if m_common else None

    else:
        # --- Try common-name-only prompt ---
        m = prompt_common_re.search(chunk)
        if m:
            current_prompt = m.group(1).strip()
            current_sci = None

            m_common = common_name_re.search(current_prompt)
            current_common = m_common.group(1).strip() if m_common else None

    # --- Extract answer ---
    m2 = re.search(r"\[FINAL ANSWER\](.*)", chunk, flags=re.I | re.S)
    if m2 and current_prompt is not None:
        answer = m2.group(1).strip()

        records.append({
            "prompt": current_prompt,
            "common_name": current_common,
            "scientific_name": current_sci,
            "answer": answer,
        })

        # reset
        current_prompt = None
        current_common = None
        current_sci = None

# # Extract flavor descriptors using the defined function
# all_ing = extract_flavor_descriptors(records, combined_flavor_descriptors)
# all_ing[:1], len(all_ing)
